In [1]:
import torch
from peft import PeftModel, PeftConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig
from pathlib import Path

In [2]:
content_dir = Path('.').resolve()
MODEL_NAME = content_dir / 'output'

In [3]:
MODEL_NAME = content_dir / 'output'
DEFAULT_MESSAGE_TEMPLATE = "<s>{role}\n{content}</s>\n"
DEFAULT_SYSTEM_PROMPT = "Ты — ruGPT-3.5, русскоязычный автоматический ассистент. Ты разговариваешь с людьми и помогаешь им."

In [4]:
class Conversation:
    def __init__(
            self,
            message_template=DEFAULT_MESSAGE_TEMPLATE,
            system_prompt=DEFAULT_SYSTEM_PROMPT,
            start_token_id=2,
            bot_token_id=46787
    ):
        self.message_template = message_template
        self.start_token_id = start_token_id
        self.bot_token_id = bot_token_id
        self.messages = [{
            "role": "system",
            "content": system_prompt
        }]

    def get_start_token_id(self):
        return self.start_token_id

    def get_bot_token_id(self):
        return self.bot_token_id

    def add_user_message(self, message):
        self.messages.append({
            "role": "user",
            "content": message
        })

    def add_bot_message(self, message):
        self.messages.append({
            "role": "bot",
            "content": message
        })

    def get_prompt(self, tokenizer):
        final_text = ""
        for message in self.messages:
            message_text = self.message_template.format(**message)
            final_text += message_text
        final_text += tokenizer.decode([self.start_token_id, self.bot_token_id])
        return final_text.strip()

In [5]:
def generate(model, tokenizer, prompt, generation_config):
    data = tokenizer(prompt, return_tensors="pt")
    data = {k: v.to(model.device) for k, v in data.items()}
    output_ids = model.generate(
        **data,
        generation_config=generation_config
    )[0]
    output_ids = output_ids[len(data["input_ids"][0]):]
    output = tokenizer.decode(output_ids, skip_special_tokens=True)
    return output.strip()

In [6]:
# Load base model
config = PeftConfig.from_pretrained(str(MODEL_NAME))
model = AutoModelForCausalLM.from_pretrained(
    config.base_model_name_or_path,
    load_in_8bit=True,
    torch_dtype=torch.float16,
    device_map="cuda:0"
)
model = PeftModel.from_pretrained(
    model,
    MODEL_NAME,
    torch_dtype=torch.float16
)
model.eval()

Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
generation_config = GenerationConfig.from_pretrained(MODEL_NAME)
print(generation_config)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


GenerationConfig {
  "bos_token_id": 2,
  "do_sample": true,
  "eos_token_id": 3,
  "max_new_tokens": 1536,
  "no_repeat_ngram_size": 15,
  "pad_token_id": 0,
  "repetition_penalty": 1.15,
  "temperature": 0.2,
  "top_k": 30,
  "top_p": 0.9
}



In [8]:
# Start conversation
conversation = Conversation()
while True:
    user_message = input("User: ")
    if user_message.strip() == "/reset":
        conversation = Conversation()
        print("History reset completed!")
        continue
    conversation.add_user_message(user_message)
    prompt = conversation.get_prompt(tokenizer)
    output = generate(
        model=model,
        tokenizer=tokenizer,
        prompt=prompt,
        generation_config=generation_config
    )
    conversation.add_bot_message(output)
    print("ruGPT-3.5:", output)
    print()
    print("==============================")
    print()

ruGPT-3.5: bot
Я - поэт! Я - поэт! И я пишу стихи! 

И вот однажды я решил написать стих про любовь. Но у меня ничего не получалось. Тогда я взял листок бумаги и начал писать: "Любовь... Любовь..." И тут вдруг мне в голову пришла мысль: "А что такое любовь?" И я задумался над этим вопросом. 

Но потом я вспомнил о своей девушке и подумал: "Если бы она была моей девушкой, то я любил бы ее". И тогда я понял, что любовь - это когда ты любишь кого-то.


ruGPT-3.5: И вот однажды я решил написать стих про любовь. Но у меня ни чего не получалось. Тогда я взял лист бумаги и начал писать: "Любовь... Любовь...". И тут вдруг мне в голову пришла мысль: "Что такое любовь?". И я задумался над этим вопросом. 

Но потом я вспомнил о своем парне и подумал: "Если бы он был моим парнем, то я любил бы его". И тогда я понял, что любовь - это когда ты любишь кого то.


ruGPT-3.5: Конечно же, мужчина должен брать кредит на два миллиона рублей. Ведь если он будет жить по средствам, то ему придется экономить к

KeyboardInterrupt: 